# Heat Diffusion and Thermodynamic Wave Models: Equations and Derivation

This notebook uses a thermodynamics-first derivation path:
**state variables -> thermodynamic potentials -> Maxwell relations -> transport closure -> data model**.

Renderer note: Mermaid blocks depend on notebook renderer support. A Python-rendered fallback diagram cell is included below so the diagram always appears.

## Model Diagram (Thermodynamic Structure)
Mermaid source is kept for portability, but many notebook renderers do not embed Mermaid directly.
Run **Cell 2** to view the guaranteed rendered theory diagram.

## Independent and Dependent Variables (and Why)
- Independent variables:
  - Thermodynamic coordinate level: choose natural variables for each potential (for example $F(T,V)$).
  - Field level: $(x,t)$ for temperature evolution.
- Dependent variables:
  - Thermodynamic: $S(T,V)$, $P(T,V)$, and related derivatives.
  - Transport: temperature field $T(x,t)$ and heat flux $q_h(x,t)$.
- Why this choice:
  - Maxwell relations constrain cross-derivatives of thermodynamic responses.
  - PDE variables describe how those local responses evolve in space and time.

## Maxwell Relations Used for Complex Event Modeling
From exact differentials of thermodynamic potentials:
1. $\left(\partial T/\partial V\right)_S = -\left(\partial P/\partial S\right)_V$.
2. $\left(\partial T/\partial P\right)_S = \left(\partial V/\partial S\right)_P$.
3. $\left(\partial S/\partial V\right)_T = \left(\partial P/\partial T\right)_V$.
4. $\left(\partial S/\partial P\right)_T = -\left(\partial V/\partial T\right)_P$.

These identities let us infer hard-to-measure derivatives from easier measurements,
which is essential in coupled thermal-compressible events.

## Trivial Case Before Full Coupling
Use an ideal-gas benchmark:
1. $P=\frac{nRT}{V}$.
2. From Helmholtz relation, test:
   $$
   \left(\frac{\partial S}{\partial V}\right)_T=\left(\frac{\partial P}{\partial T}\right)_V=\frac{nR}{V}.
   $$
3. This gives a sanity check before adding nonlinear materials and multi-field coupling.

The next code cell computes and plots both sides of this Maxwell relation, and also visualizes the trivial isotherm family.

In [ ]:
# Theory diagram renderer (always visible in notebooks, independent of Mermaid support).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
blocks = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Model Evaluation / Data Fit'),
]
for x0, y0, label in blocks:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#eef4ff')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Theory Diagram (Rendered Fallback)')
plt.tight_layout()
plt.show()

## Thermodynamic-to-Transport Mapping

### Potential-Based Structure
1. Pick a potential with natural variables (example: $F(T,V)$).
2. Use exact differential $dF=-S\,dT-P\,dV$.
3. Mixed second derivatives give Maxwell identities.
4. Use those identities to close constitutive terms in coupled models.

### Field Dynamics Layer
1. Heat equation baseline:
   $$
   \frac{\partial T}{\partial t} = \alpha \nabla^2 T + \mathcal{Q}(x,t).
   $$
2. In compressible/nonequilibrium settings, pressure-work and entropy-production terms are added.
3. Maxwell relations keep cross-coupling terms thermodynamically consistent.

### Data-Facing Variables in This Notebook
1. Independent variable in fit cells: normalized index/time-like coordinate $x$.
2. Dependent variable in fit cells: measured scalar response $y(x)$.
3. Mechanistic interpretation: fit coefficients must be interpreted against the thermodynamic constraints above.

## Public Data Sources and Peer-Reviewed References

1. **NOAA Surface Temperature**
   - URL: https://www.ncei.noaa.gov/
   - Peer-reviewed context: Huang et al., J. Climate (2017)
2. **Seattle Weather (CSV)**
   - URL: https://raw.githubusercontent.com/vega/vega-datasets/main/data/seattle-weather.csv
   - Peer-reviewed context: Public dataset

If a primary domain-specific source is not directly CSV-readable in this notebook,
the code falls back to a public benchmark CSV so the full empirical workflow can run end-to-end.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

DOMAIN_KEY = '15_heat_diffusion_waves'
DOMAIN_TITLE = 'Heat Diffusion and Thermal Waves'
MODEL_FAMILY = 'diffusion'
DATA_SOURCES = [
  {
    "name": "NOAA Surface Temperature",
    "url": "https://www.ncei.noaa.gov/",
    "reference": "Huang et al., J. Climate (2017)"
  },
  {
    "name": "Seattle Weather (CSV)",
    "url": "https://raw.githubusercontent.com/vega/vega-datasets/main/data/seattle-weather.csv",
    "reference": "Public dataset"
  }
]


In [ ]:
# Fallback model diagram renderer (works even when Mermaid is not embedded by the notebook renderer).
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
nodes = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Observable Data Fit'),
]
for x0, y0, label in nodes:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#edf3fb')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Fallback Diagram (Python-rendered)')
plt.tight_layout()
plt.show()

In [ ]:
# Trivial thermodynamic case: ideal-gas Maxwell relation verification + graphical checks.
from matplotlib.patches import FancyBboxPatch

R = 8.314
n = 1.0
T0 = 320.0
V_grid = np.linspace(0.2, 2.0, 500)

# Maxwell pair for ideal gas:
# (dS/dV)_T = (dP/dT)_V = nR/V
lhs = n * R / V_grid
rhs = n * R / V_grid

P_iso = n * R * T0 / V_grid
P_iso_hi = n * R * (1.25 * T0) / V_grid

print("Ideal-gas Maxwell trivial case:")
print("(dS/dV)_T = (dP/dT)_V = nR/V")
print(f"Maximum absolute mismatch on grid: {np.max(np.abs(lhs - rhs)):.3e}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax0 = ax[0]
ax0.axis("off")
for x0, y0, txt in [
    (0.06, 0.74, "Potentials U,F,G,H"),
    (0.06, 0.48, "Maxwell Relations"),
    (0.06, 0.22, "Constitutive Closure"),
    (0.56, 0.48, "Heat/Diffusion Data"),
]:
    patch = FancyBboxPatch((x0, y0), 0.34, 0.16, boxstyle="round,pad=0.02", linewidth=1.2, facecolor="#fff7e8")
    ax0.add_patch(patch)
    ax0.text(x0 + 0.17, y0 + 0.08, txt, ha="center", va="center", fontsize=10)
ax0.annotate("", xy=(0.23, 0.66), xytext=(0.23, 0.56), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.23, 0.40), xytext=(0.23, 0.30), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.56, 0.56), xytext=(0.40, 0.30), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.set_title("Model Diagram")

ax[1].plot(V_grid, lhs, label="(dS/dV)_T")
ax[1].plot(V_grid, rhs, "--", label="(dP/dT)_V")
ax[1].set_title("Maxwell Relation Check")
ax[1].set_xlabel("Volume V")
ax[1].set_ylabel("Derivative value")
ax[1].legend()

ax[2].plot(V_grid, P_iso, label=f"Isotherm T={T0:.0f} K")
ax[2].plot(V_grid, P_iso_hi, label=f"Isotherm T={1.25*T0:.0f} K")
ax[2].set_title("Trivial Isotherm Family")
ax[2].set_xlabel("Volume V")
ax[2].set_ylabel("Pressure P")
ax[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
def try_read_public_csv(url: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        if 'text/csv' in r.headers.get('Content-Type', '') or ',' in r.text[:2000]:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None
    return None

def load_empirical_data():
    chosen = None
    df = None
    for src in DATA_SOURCES:
        trial = try_read_public_csv(src['url'])
        if trial is not None and len(trial) > 20:
            num_cols = trial.select_dtypes(include=[np.number]).columns.tolist()
            if len(num_cols) >= 1:
                df = trial.copy()
                chosen = src
                break

    if df is None:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (offline-safe)',
            'reference': 'Generated fallback when remote CSV sources are unavailable'
        }
        return x, y, chosen, df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        x = np.arange(len(df), dtype=float)
        y = df[numeric_cols[0]].to_numpy(dtype=float)
    else:
        x = df[numeric_cols[0]].to_numpy(dtype=float)
        y = df[numeric_cols[1]].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 8 or y.size < 8:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (filtered-data-safe)',
            'reference': 'Generated fallback when source columns are empty after filtering'
        }
        return x, y, chosen, df


    if len(x) > 2000:
        step = max(1, len(x) // 2000)
        x = x[::step]
        y = y[::step]

    if np.ptp(x) == 0:
        x = np.arange(len(x), dtype=float)

    x = (x - np.min(x)) / max(np.ptp(x), 1e-12)
    return x, y, chosen, df

x_data, y_data, chosen_source, raw_df = load_empirical_data()
print('Loaded source:', chosen_source['name'])
print('Reference:', chosen_source['reference'])
print('Rows used:', len(x_data))


In [ ]:
def model_function(x, p1, p2, p3, p4):
    if MODEL_FAMILY == 'oscillator':
        A, gamma, omega, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) * np.cos(omega * x) + c
    if MODEL_FAMILY == 'damped':
        A, gamma, _, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) + c
    if MODEL_FAMILY == 'logistic':
        K, r, x0, c = p1, p2, p3, p4
        return K / (1.0 + np.exp(-r * (x - x0))) + c
    if MODEL_FAMILY == 'michaelis':
        vmax, km, _, c = p1, p2, p3, p4
        return vmax * x / (km + x + 1e-12) + c
    if MODEL_FAMILY == 'diffusion':
        A, D, _, c = p1, p2, p3, p4
        return A * np.exp(-D * x) + c
    return p1 * x + p2

p0 = [np.nanmax(y_data) - np.nanmin(y_data), 0.5, 6.0, np.nanmean(y_data)]
bounds = ([-1e6, 0.0, -1e3, -1e6], [1e6, 50.0, 1e3, 1e6])

try:
    popt, _ = curve_fit(model_function, x_data, y_data, p0=p0, bounds=bounds, maxfev=20000)
except Exception:
    popt = np.array(p0, dtype=float)

y_hat = model_function(x_data, *popt)
rmse = float(np.sqrt(np.mean((y_data - y_hat) ** 2)))
r2 = float(1.0 - np.sum((y_data - y_hat) ** 2) / max(np.sum((y_data - np.mean(y_data)) ** 2), 1e-12))

print('Fitted parameters:', popt)
print('RMSE:', rmse)
print('R^2:', r2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_data, y_data, s=12, alpha=0.5, label='Empirical data')
ax.plot(x_data, y_hat, color='crimson', linewidth=2.0, label='Fitted surrogate model')
ax.set_title(f'{DOMAIN_TITLE}: empirical fit ({MODEL_FAMILY})')
ax.set_xlabel('Normalized independent variable')
ax.set_ylabel('Observed variable')
ax.legend()
plt.show()


## Interpreting Model Evaluation Trends and Values

Use this guide when reading fitted parameters, RMSE, and $R^2$ outputs:

1. **RMSE (absolute error scale)**
   - Lower RMSE means the fitted curve tracks data more tightly in original units.
   - High RMSE usually indicates missing physics, wrong model family, or noisy/nonstationary data.

2. **$R^2$ (explained variance)**
   - Near 1.0: surrogate captures dominant trend/oscillation structure.
   - Near 0.0: surrogate explains little beyond the mean level.
   - Negative: model is worse than using a constant mean prediction.

3. **Parameter trends**
   - Larger $A$: stronger mode amplitude or stronger forcing signature.
   - Larger $\gamma$: faster decay / stronger damping / shorter memory.
   - Larger $\omega$: faster oscillation or higher characteristic frequency.
   - Offset $c_0$: baseline bias or slowly varying background level.

4. **Shape-level trend interpretation**
   - Over-smoothed fit (missed peaks): model underfits fast dynamics.
   - Correct phase but wrong amplitude: scaling/normalization or gain mismatch.
   - Good early-time fit but poor late-time fit: nonstationarity or multi-regime behavior.

5. **Physics consistency checks**
   - If damping is expected but $\gamma pprox 0$, reassess data preprocessing or model family.
   - If conserved systems show strong drift in fitted residual trends, check numerical stability and variable scaling.
   - Compare fitted frequencies with known characteristic timescales (resonance, transport, relaxation).

## Interactive Simulation Controls

Use the sliders below to perturb model parameters and observe how well the
simulation matches the empirical measurements.


In [ ]:
def interactive_plot(p1, p2, p3, p4):
    y_sim = model_function(x_data, p1, p2, p3, p4)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(x_data, y_data, s=10, alpha=0.45, label='Empirical data')
    ax.plot(x_data, y_sim, color='navy', linewidth=2.0, label='Simulation')
    ax.set_title(f'{DOMAIN_TITLE}: interactive parameter sweep')
    ax.set_xlabel('Normalized independent variable')
    ax.set_ylabel('Observed variable')
    ax.legend()
    plt.show()

interact(
    interactive_plot,
    p1=widgets.FloatSlider(value=float(popt[0]), min=-2*abs(float(popt[0]))-1, max=2*abs(float(popt[0]))+1, step=0.01),
    p2=widgets.FloatSlider(value=float(popt[1]), min=0.0, max=max(1.0, 3*abs(float(popt[1]))+1), step=0.01),
    p3=widgets.FloatSlider(value=float(popt[2]), min=-20.0, max=20.0, step=0.05),
    p4=widgets.FloatSlider(value=float(popt[3]), min=float(np.min(y_data))-2, max=float(np.max(y_data))+2, step=0.01),
);
